# Week 2.1 | Which DPP-4 inhibitors would you compare?

A structural biologist can compare three drug–protein complexes. You have six medicines that inhibit human dipeptidyl peptidase 4 (DPP-4). **Which three would give you a varied set of molecular structures to look at?**

Choose three from the drawings, then check your choice using calculated properties and structural similarity. You will finish with three drug IDs, one feature from the drawings and one similarity score that supports your choice.

| Practical | What you will do |
|---|---|
| 1. Inspect six drugs | Pick three from their drawings |
| 2. Represent a molecule | Compare a molecular graph, SMILES, SELFIES and InChI |
| 3. Compare properties | Read one value for each of two drugs |
| 4. Compare fingerprints | See whether MACCS and Morgan find the same closest drug |
| 5. Make your shortlist | Compare the three pairs in your selection |

**In Colab.** Choose **File → Save a copy in Drive** to keep your work. Run the cells in order with the play button on the left. When a step asks you to change a value, keep the quotation marks and run that cell again. Double-click a text cell to write your answers.

**Optional GenAI help.** For a longer cell, click **Show code** and paste the code with the short question next to it. Run the cell and check one part of the explanation against the output. You can also do this with a partner.

### Set up Colab

Run the two cells below. The first installs **RDKit** for reading and drawing molecules and **SELFIES** for converting molecule strings. The second loads the six drugs. When you see “Six drugs are ready”, continue to Section 1.

In [ ]:
# @title Install RDKit and SELFIES { display-mode: "form" }
%pip -q install rdkit selfies

In [ ]:
# @title Load the six drugs { display-mode: "form" }
from itertools import combinations

import pandas as pd
import selfies as sf
from IPython.display import display
from rdkit import Chem, DataStructs
from rdkit.Chem import Crippen, Descriptors, Draw, Lipinski, MACCSkeys, rdFingerprintGenerator, rdMolDescriptors

drugs = pd.DataFrame([
    ('D01', 'Vildagliptin', 'C1C[C@H](N(C1)C(=O)CNC23CC4CC(C2)CC(C4)(C3)O)C#N', '3W2T'),
    ('D02', 'Saxagliptin', 'C1[C@@H]2C[C@@H]2N([C@@H]1C#N)C(=O)[C@H](C34CC5CC(C3)CC(C5)(C4)O)N', '3BJM'),
    ('D03', 'Alogliptin', 'CN1C(=O)C=C(N(C1=O)CC2=CC=CC=C2C#N)N3CCC[C@H](C3)N', '3G0B'),
    ('D04', 'Linagliptin', 'CC#CCN1C2=C(N=C1N3CCC[C@H](C3)N)N(C(=O)N(C2=O)CC4=NC5=CC=CC=C5C(=N4)C)C', '2RGU'),
    ('D05', 'Sitagliptin', 'N[C@@H](CC(=O)N1CCn2c(C1)nnc2C(F)(F)F)Cc3cc(F)c(F)cc3F', '1X70'),
    ('D06', 'Teneligliptin', 'CC1=NN(C(=C1)N2CCN(CC2)[C@H]3C[C@H](NC3)C(=O)N4CCSC4)C5=CC=CC=C5', '3VJK'),
], columns=["ID", "Drug", "SMILES", "PDB"]).set_index("ID")
molecules = drugs["SMILES"].map(Chem.MolFromSmiles)
print("Six drugs are ready.")

## 1. Inspect six drugs

In a 2D drawing, an unlabelled corner or line end usually represents a carbon atom. Lines show bonds, and letters such as N, O and F label other elements. Look for rings, branches and labelled atoms as you compare the drawings.

In [ ]:
# @title Draw the six inhibitors { display-mode: "form" }
labels = [f"{drug_id} {name}" for drug_id, name in drugs["Drug"].items()]
display(Draw.MolsToGridImage(list(molecules), legends=labels, molsPerRow=3, subImgSize=(260, 185)))

**Discuss.** Choose two drugs that share a feature you can point to in both drawings. Then pick three drugs that look different enough to compare. Keep your first shortlist for Section 5.

- Similar pair and shared feature …
- First three IDs …

## 2. Represent a molecule

The six drawings in Section 1 came from molecular structures stored as text. Ethanol is small enough to see how the same structure can be written in several ways.

| Representation | What it shows |
|---|---|
| **Molecular graph** | Atoms are points and bonds connect them. You can trace which atoms are neighbours. |
| **SMILES** | A short text description of a structure. `CCO` traces two carbons followed by oxygen. |
| **SELFIES** | A text format that uses bracketed pieces called tokens. For ethanol, these are `[C][C][O]`. |
| **InChI** | The International Chemical Identifier. It gives a standard text identifier for matching chemical records. Look for the molecular formula `C2H6O` in the output. |

**Try it.** Run the cell to see the labelled graph and the usual 2D drawing. The labelled graph shows the non-hydrogen atoms. Trace the two carbons and the oxygen in both drawings, then find them in the SMILES and SELFIES strings.

In [ ]:
# @title Show the four representations of ethanol { display-mode: "form" }
smiles = "CCO"
ethanol = Chem.MolFromSmiles(smiles)
labelled_graph = Chem.Mol(ethanol)
for atom in labelled_graph.GetAtoms():
    atom.SetProp("atomLabel", atom.GetSymbol())

display(Draw.MolsToGridImage(
    [labelled_graph, ethanol],
    legends=["Labelled molecular graph", "Standard 2D drawing"],
    molsPerRow=2,
    subImgSize=(250, 160)
))
print("SMILES:", smiles)
print("SELFIES:", sf.encoder(smiles))
print("InChI:", Chem.MolToInchi(ethanol))

**Optional GenAI prompt.** “Use ethanol to explain what a molecular graph, SMILES, SELFIES and InChI each show. Give one short sentence for each.”

### Can two SMILES describe the same molecule?

A SMILES string can also record stereochemistry. RDKit reads a SMILES string as a structure and can write it back as a **canonical SMILES**, a consistent text representation under the same settings.

| SMILES | What it describes |
|---|---|
| `CCO` and `OCC` | Ethanol written from opposite ends |
| `COC` | Dimethyl ether, with oxygen between the two carbons |

**Try it.** Run the cell, then change only `"OCC"` to `"COC"` and run it again. Which comparisons change?

In [ ]:
other_smiles = "OCC"

ethanol = Chem.MolFromSmiles("CCO")
other_molecule = Chem.MolFromSmiles(other_smiles)
print("Same text?", "CCO" == other_smiles)
print("Same canonical SMILES?", Chem.MolToSmiles(ethanol) == Chem.MolToSmiles(other_molecule))
print("Same InChI?", Chem.MolToInchi(ethanol) == Chem.MolToInchi(other_molecule))
display(Draw.MolsToGridImage(
    [ethanol, other_molecule], legends=["CCO", other_smiles], molsPerRow=2, subImgSize=(180, 135)
))

**Check your result.** `CCO` and `OCC` give different input strings and match on canonical SMILES and InChI. In `COC`, the oxygen sits between the carbons, so both structure-based comparisons change.

**Optional GenAI prompt.** Copy the question below into a chat. Check the explanation against the drawings and the printed comparisons. You can explain the result to a partner using the same evidence.

```text
Why does Python report different text for CCO and OCC even though both describe ethanol? Explain in two sentences.
```

## 3. Compare properties

A **molecular descriptor** is a number that summarises one property. Compare the pair from Section 1 using the values in the table. Vildagliptin (`D01`) and saxagliptin (`D02`) are the worked example.

| Descriptor | What it describes |
|---|---|
| MW (g/mol) | Molecular weight |
| logP | Estimated preference for octanol over water. Higher values indicate a greater preference for octanol. |
| TPSA (Å²) | Estimated polar surface area based on the molecular structure |
| HBD | Number of hydrogen-bond donor sites |
| HBA | Number of hydrogen-bond acceptor sites |

In [ ]:
# @title Calculate the five descriptors { display-mode: "form" }
def calculate_properties(structures):
    return pd.DataFrame({
        "MW (g/mol)": structures.map(Descriptors.MolWt),
        "logP": structures.map(Crippen.MolLogP),
        "TPSA (Å²)": structures.map(rdMolDescriptors.CalcTPSA),
        "HBD": structures.map(Lipinski.NumHDonors),
        "HBA": structures.map(Lipinski.NumHAcceptors),
    })

properties = calculate_properties(molecules)

**Try it.** Run the example. Then replace the IDs in quotes with the two drugs you found similar. Change one ID at a time. Read the **same column** for both drugs.

**Discuss.** Which descriptor supports your choice? Give both values. What difference in the drawings does that descriptor miss?

**Optional GenAI prompt for Calculate the five descriptors.** “What goes into this code, and what does it return? Use two short sentences.”

In [ ]:
first_drug = "D01"
second_drug = "D02"

pair = [first_drug, second_drug]
drugs[["Drug"]].join(properties).loc[pair].round(2)

## 4. Compare fingerprints

A molecular **fingerprint** records features of a structure. In the binary fingerprints used here, a bit is on (1) when the fingerprint records a feature.

| Term | How it works |
|---|---|
| Substructure-based fingerprint | Records patterns within a molecule. A common approach is to check every molecule against the same list of predefined patterns. These are called structural keys. |
| MACCS fingerprint | A structural-key fingerprint. The public set defines 166 keys. Each key checks a predefined feature, such as a pattern of bonded atoms. |
| Circular fingerprint | Builds features around each atom by following bonds outward. The radius tells you how many bonds away to look. |
| Morgan fingerprint | A circular fingerprint. Here we look up to two bonds from each atom (radius 2) and store the resulting features in 2,048 bits. The settings used here ignore stereochemistry. |

For example, start at the oxygen in ethanol. One bond takes you to the adjacent carbon. Two bonds take you to the other carbon.

**Try it on paper.** Use two simple structural keys: “Is there a C–C bond?” and “Is there a C–O bond?” Write a 1 for yes and a 0 for no. Which two-bit fingerprint would you give `CCO`, `OCC` and `COC`? These two keys are an illustration of how a checklist works.

### Compare two fingerprints

**Tanimoto similarity** divides the number of bits on in **both** fingerprints by the number on in **at least one**. A higher score means the two fingerprints share a larger fraction of their on-bits.

| Toy fingerprint | Bit 1 | Bit 2 | Bit 3 | Bit 4 | Bit 5 |
|---|---:|---:|---:|---:|---:|
| A | 1 | 0 | 1 | 1 | 0 |
| B | 1 | 1 | 0 | 1 | 0 |

**Predict.** Which bits are on in both? Which are on in at least one? Count them before you run the calculation.

In [ ]:
on_in_both = 2
on_in_either = 4
print("Tanimoto similarity:", on_in_both / on_in_either)

### Check what happens when the SMILES changes

**Try it.** Run the two cells below. Then change `"OCC"` to `"COC"`. Does the MACCS bit for an oxygen bonded to hydrogen change? What happens to the two similarity scores?

RDKit calculates both fingerprints from the structure it reads from SMILES. A score of 1 means the on-bits match for that fingerprint. Different molecules can share the same fingerprint.

In [ ]:
# @title Prepare MACCS and Morgan fingerprints { display-mode: "form" }
morgan = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
maccs_fingerprints = molecules.map(MACCSkeys.GenMACCSKeys)
fingerprints = molecules.map(morgan.GetFingerprint)

def show_smiles_scores(smiles):
    ethanol = Chem.MolFromSmiles("CCO")
    other = Chem.MolFromSmiles(smiles)
    ethanol_maccs = MACCSkeys.GenMACCSKeys(ethanol)
    other_maccs = MACCSkeys.GenMACCSKeys(other)
    print("Same canonical SMILES?", Chem.MolToSmiles(ethanol) == Chem.MolToSmiles(other))
    print(f"MACCS key 139 (oxygen bonded to H): CCO = {int(ethanol_maccs[139])}, {smiles} = {int(other_maccs[139])}")
    maccs_score = DataStructs.TanimotoSimilarity(ethanol_maccs, other_maccs)
    morgan_score = DataStructs.TanimotoSimilarity(morgan.GetFingerprint(ethanol), morgan.GetFingerprint(other))
    print(f"MACCS similarity: {maccs_score:.3f}")
    print(f"Morgan similarity: {morgan_score:.3f}")

In [ ]:
comparison_smiles = "OCC"
show_smiles_scores(comparison_smiles)

### Find the closest drug with MACCS and Morgan

A **reference drug** is the one you compare with the others. Start with sitagliptin (`D05`). Look back at the drawings and predict which drug will rank first. Run both cells below. Each table ranks the other five drugs. The reference drug and Tanimoto calculation are the same in both tables.

**Optional GenAI prompt for Prepare the similarity search.** “How does this code find the closest drug? Explain in three short steps.”

In [ ]:
# @title Prepare the similarity search { display-mode: "form" }
def rank_neighbours(reference_id, library, library_fingerprints):
    candidates = library_fingerprints.drop(reference_id)
    scores = DataStructs.BulkTanimotoSimilarity(
        library_fingerprints.loc[reference_id], list(candidates)
    )
    matches = library.loc[candidates.index, ["Drug"]].copy()
    matches["Tanimoto similarity"] = scores
    return matches.sort_values("Tanimoto similarity", ascending=False, kind="stable")

def show_neighbours(reference_id):
    maccs_ranking = rank_neighbours(reference_id, drugs, maccs_fingerprints)
    morgan_ranking = rank_neighbours(reference_id, drugs, fingerprints)
    print("MACCS ranking")
    display(maccs_ranking.round(4))
    print("Morgan ranking")
    display(morgan_ranking.round(4))
    maccs_match = maccs_ranking.index[0]
    morgan_match = morgan_ranking.index[0]
    display(Draw.MolsToGridImage(
        list(molecules.loc[[reference_id, maccs_match, morgan_match]]),
        legends=[
            f"Reference: {drugs.at[reference_id, 'Drug']}",
            f"MACCS: {drugs.at[maccs_match, 'Drug']}",
            f"Morgan: {drugs.at[morgan_match, 'Drug']}",
        ],
        molsPerRow=3,
        subImgSize=(250, 175)
    ))

In [ ]:
reference_drug = "D05"
show_neighbours(reference_drug)

**Try it.** Replace `"D05"` with `"D06"` to use teneligliptin as the reference. Run the ranking cell again.

**Discuss.** Which drug ranks first with MACCS and which ranks first with Morgan? Find one feature in the drawings that teneligliptin shares with either match. Use the order within each table to compare the rankings.

## 5. Make your shortlist

The ranking in Section 4 compared each drug with a single reference. Now compare **all three pairs** within your shortlist using the Morgan fingerprints. For three drugs A, B and C, those pairs are A–B, A–C and B–C.

**Try it.** Run both cells. Then replace the three quoted IDs in the second cell with your first shortlist. The table shows the most similar pair at the top. If you want more variety, try replacing one member of that pair and check whether the highest score goes down.

**Optional GenAI prompt for Prepare the shortlist comparison.** “Which pairs does this code compare if I give it D01, D02 and D05?”

In [ ]:
# @title Prepare the shortlist comparison { display-mode: "form" }
def show_shortlist(selected_drugs):
    if len(selected_drugs) != 3 or len(set(selected_drugs)) != 3 or not set(selected_drugs) <= set(drugs.index):
        print("Choose three different IDs from D01 to D06.")
    else:
        comparisons = [
            (f"{left} {drugs.at[left, 'Drug']}", f"{right} {drugs.at[right, 'Drug']}",
             DataStructs.TanimotoSimilarity(fingerprints.loc[left], fingerprints.loc[right]))
            for left, right in combinations(selected_drugs, 2)
        ]
        pair_scores = pd.DataFrame(comparisons, columns=["First drug", "Second drug", "Tanimoto similarity"])
        display(pair_scores.sort_values("Tanimoto similarity", ascending=False).round(3))

In [ ]:
selected_drugs = ["D01", "D02", "D05"]
show_shortlist(selected_drugs)

**Your final shortlist**

- Three drug IDs …
- Most similar pair in my shortlist and its score …
- One visible structural difference that supports my choice …
- I kept or changed my first selection because …

### From a 2D drawing to a bound structure

We used the drawings and fingerprints to compare the drug molecules. To see where a drug sits in DPP-4, we need a 3D model of the complex. Open the **3D View** of [PDB entry 1X70](https://www.rcsb.org/structure/1X70) to see the experimental model with sitagliptin bound.

**For Week 2.2.** Bring your shortlist. You will compare it with groups obtained from a larger collection and look at how the selected compounds interact with DPP-4. What would you check in two complexes to see whether the drugs sit in the same part of the enzyme?

## Optional extensions

### Can a fingerprint tell mirror images apart?

These two SMILES describe mirror-image forms of lactic acid. The atoms connect in the same order. The `@` and `@@` labels indicate opposite configurations at the same carbon in these two strings.

**Try it.** Run the cell. Do the canonical SMILES match? Do the fingerprints match with the settings we used? What changes when Morgan includes stereochemistry?

In [ ]:
# @title Compare the two forms of lactic acid { display-mode: "form" }
first = Chem.MolFromSmiles("C[C@H](O)C(=O)O")
second = Chem.MolFromSmiles("C[C@@H](O)C(=O)O")
morgan_with_stereo = rdFingerprintGenerator.GetMorganGenerator(
    radius=2, fpSize=2048, includeChirality=True
)
print("Same canonical SMILES?", Chem.MolToSmiles(first) == Chem.MolToSmiles(second))
for label, make_fingerprint in [
    ("MACCS", MACCSkeys.GenMACCSKeys),
    ("Morgan", morgan.GetFingerprint),
    ("Morgan with stereochemistry", morgan_with_stereo.GetFingerprint),
]:
    score = DataStructs.TanimotoSimilarity(make_fingerprint(first), make_fingerprint(second))
    print(f"{label} similarity: {score:.3f}")

### Measure a distance in the 1X70 complex

In the X-ray structure of DPP-4 with sitagliptin, the ligand and protein atoms have coordinates in the same 3D frame. The code below uses coordinates for three atoms from chain A of the deposited model. `N20` is an atom in sitagliptin. `OE1` and `OE2` are the two side-chain oxygen atoms in the protein residue Glu205.

**Try it.** Run the cell. Then replace `"OE1"` with `"OE2"` and run it again. Which oxygen is closer to the drug atom? Why would a 2D drawing of sitagliptin alone be insufficient for this measurement?

| File | What it contains |
|---|---|
| Sitagliptin SDF from PubChem | A 2D layout of sitagliptin |
| Sitagliptin SDF from 1X70 | The 3D coordinates of the bound ligand |
| 1X70 PDB file | The ligand and the protein in the same 3D coordinate frame |

[PubChem 2D SDF](https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/4369359/SDF?record_type=2d) · [Bound ligand SDF](https://models.rcsb.org/v1/1x70/ligand?auth_asym_id=A&label_comp_id=715&encoding=sdf) · [Full 1X70 PDB file](https://files.rcsb.org/download/1X70.pdb)

In [ ]:
oxygen = "OE1"

from math import dist

# Coordinates in ångströms from 1X70, chain A.
ligand_n20 = (38.203, 51.846, 34.353)
glu205 = {'OE1': (38.443, 56.65, 35.021), 'OE2': (38.201, 54.481, 35.372)}

distance = dist(ligand_n20, glu205[oxygen])
print(f"N20 to Glu205 {oxygen}: {distance:.2f} Å")

### Search the larger collection

Your first ranking used only the six reference drugs. The course collection contains 22 more ligands from DPP-4 structures. This cell loads the full table and finds the five closest structures to sitagliptin. Compare the top score with sitagliptin’s top score from Section 4. Which ligand gives the higher score?

[Week 2 course data and provenance](https://github.com/tcobanoglu/dadd-course-data/tree/main/data/week2)

In [ ]:
# @title Search all 28 compounds { display-mode: "form" }
course_csv = "https://raw.githubusercontent.com/tcobanoglu/dadd-course-data/main/data/week2/dpp4_compounds.csv"
collection = pd.read_csv(course_csv, index_col="compound_id").rename(columns={"name": "Drug"})
collection_molecules = collection["smiles"].map(Chem.MolFromSmiles)
collection_fingerprints = collection_molecules.map(morgan.GetFingerprint)
rank_neighbours("D05", collection, collection_fingerprints).head(5).round(3)

### Read a radar plot

The table in Section 3 shows the original values. A radar plot puts these descriptors on separate spokes and scales each one from 0 to 1 using the smallest and largest values in the 28-compound collection. Run **Search all 28 compounds** first. This plot uses the pair you chose in Section 3.

**Discuss.** Suppose a new compound has a higher TPSA than all 28 compounds. What would happen to your two drugs’ positions on the TPSA spoke after rescaling? What TPSA values would still appear in the table?

**Optional GenAI prompt for Plot the five scaled descriptors.** “What do 0 and 1 mean on a spoke? Use one example.”

In [ ]:
# @title Plot the five scaled descriptors { display-mode: "form" }
import matplotlib.pyplot as plt
import numpy as np

collection_properties = calculate_properties(collection_molecules)
minimum = collection_properties.min()
maximum = collection_properties.max()
scaled_pair = (properties.loc[pair] - minimum) / (maximum - minimum)

angles = np.linspace(0, 2 * np.pi, len(properties.columns), endpoint=False)
closed_angles = np.append(angles, angles[0])
figure, axis = plt.subplots(figsize=(5.5, 4.5), subplot_kw={"projection": "polar"}, layout="constrained")
for drug_id, values in scaled_pair.iterrows():
    axis.plot(closed_angles, np.append(values, values.iloc[0]), marker="o", label=drugs.at[drug_id, "Drug"])
axis.set_xticks(angles, ["MW", "logP", "TPSA", "HBD", "HBA"])
axis.set_ylim(0, 1.05)
axis.set_title("Descriptor ranges across 28 compounds", pad=20)
axis.legend(loc="upper left", bbox_to_anchor=(1.02, 1))
plt.show()

## References

- TeachOpenCADD [T033 Molecular representations](https://projects.volkamerlab.org/teachopencadd/talktorials/T033_molecular_representations.html), [T002 Molecular filtering](https://projects.volkamerlab.org/teachopencadd/talktorials/T002_compound_adme.html), [T004 Compound similarity](https://projects.volkamerlab.org/teachopencadd/talktorials/T004_compound_similarity.html) and [T008 Protein Data Bank](https://projects.volkamerlab.org/teachopencadd/talktorials/T008_query_pdb.html) informed the order of topics.
- [RDKit Getting Started](https://www.rdkit.org/docs/GettingStartedInPython.html) covers drawing molecules, calculating descriptors and working with fingerprints.
- [SELFIES documentation](https://selfies.readthedocs.io/en/latest/) and the [IUPAC definition of InChI](https://goldbook.iupac.org/terms/view/11456) describe two text formats for chemical structures.
- The six reference compounds and their identifiers come from the [Week 2 course table](https://github.com/tcobanoglu/dadd-course-data/blob/main/data/week2/dpp4_compounds.csv). The experimental example comes from [RCSB PDB entry 1X70](https://www.rcsb.org/structure/1X70).